In [4]:
import pandas as pd
from rapidfuzz import fuzz
from datetime import datetime
import json
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import json
import glob
import os
import re
import numpy as np

In [5]:
kg_df = pd.read_csv("people_entities.csv")

In [6]:
kg_df.head()

,person_uri,name,gender,era,floruit_lower,floruit_upper,wikidata,dib,birth_places,death_places
0,https://kg.virtualtreasury.ie/person/Brodrick_...,Allen Brodrick,https://www.w3id.org/virtual-treasury/ontology...,https://www.w3id.org/virtual-treasury/vocabula...,1623-07-28,1680-11-25,http://www.wikidata.org/entity/Q4731552,https://www.dib.ie/biography/Brodrick-Sir-Alan...,NaN,NaN
1,https://kg.virtualtreasury.ie/person/Jones_Art...,Abraham Jones,https://www.w3id.org/virtual-treasury/ontology...,https://www.w3id.org/virtual-treasury/vocabula...,1610-01-01,1670-01-07,NaN,https://www.dib.ie/biography/Jones-Arthur-a4337-A,NaN,NaN
2,https://kg.virtualtreasury.ie/person/Capel_Art...,William Capell,https://www.w3id.org/virtual-treasury/ontology...,https://www.w3id.org/virtual-treasury/vocabula...,1631-01-01,1683-07-13,http://www.wikidata.org/entity/Q2714210,https://www.dib.ie/biography/Capel-Arthur-a1454,NaN,"England, London"
3,https://kg.virtualtreasury.ie/person/Chicheste...,Art Chichester,https://www.w3id.org/virtual-treasury/ontology...,https://www.w3id.org/virtual-treasury/vocabula...,1606-06-16,1675-03-18,http://www.wikidata.org/entity/Q4798251,https://www.dib.ie/biography/Chichester-Arthur...,NaN,"Belfast, Northern Ireland"
4,https://kg.virtualtreasury.ie/person/Forbes_Ar...,William Holgate,https://www.w3id.org/virtual-treasury/ontology...,https://www.w3id.org/virtual-treasury/vocabula...,1623-01-01,1696-12-31,NaN,https://www.dib.ie/biography/Forbes-Sir-Arthur...,"Ireland, Longford","Ireland, Longford"


####Metadata-driven heuristic

In [7]:
def generate_candidates(doc_name, doc_date, doc_issued_at=None, doc_sent_to=None, doc_gender=None):
    candidates = []

    doc_name = str(doc_name).lower().strip()

    doc_places = []
    if pd.notna(doc_issued_at):
        doc_places.append(str(doc_issued_at).lower().strip())
    if pd.notna(doc_sent_to):
        doc_places.append(str(doc_sent_to).lower().strip())

    for _, row in kg_df.iterrows():

        kg_name = str(row['name']).lower().strip()
        name_score = fuzz.token_sort_ratio(doc_name, kg_name) / 100.0

        if name_score < 0.5:
            continue

        date_score = 0.5

        if pd.notna(doc_date) and pd.notna(row['floruit_lower']) and pd.notna(row['floruit_upper']):
            if row['floruit_lower'] <= doc_date <= row['floruit_upper']:
                date_score = 1.0
            else:
                date_score = 0.0

        location_score = 0.5

        kg_places = []
        if pd.notna(row.get('birth_places')):
            kg_places.append(str(row['birth_places']).lower())
        if pd.notna(row.get('death_places')):
            kg_places.append(str(row['death_places']).lower())

        if doc_places and kg_places:
            best_loc_score = 0.0
            for d_place in doc_places:
                for k_place in kg_places:
                    sim = fuzz.partial_ratio(d_place, k_place) / 100.0
                    best_loc_score = max(best_loc_score, sim)

            location_score = best_loc_score


        gender_score = 0.5
        if doc_gender and pd.notna(row.get('gender')):
            gender_score = 1.0 if doc_gender == row['gender'] else 0.0


        total_score = (
            0.5 * name_score +
            0.3 * date_score +
            0.2 * location_score
        )

        candidates.append({
            'person_uri': row['person_uri'],
            'name': row['name'],
            'name_score': round(name_score, 3),
            'date_score': round(date_score, 3),
            'gender_score': round(gender_score, 3),
            'location_score': round(location_score, 3),
            'total_score': round(total_score, 3)
        })

    candidates.sort(key=lambda x: x['total_score'], reverse=True)
    return candidates

In [ ]:
metadata_folder = "Metadata/"
ground_truth_folder = "Ground-Truth/"
output_json = "candidate_matches_all_docs_test.json"

kg_df['floruit_lower'] = pd.to_datetime(kg_df['floruit_lower'], errors='coerce')
kg_df['floruit_upper'] = pd.to_datetime(kg_df['floruit_upper'], errors='coerce')

all_results = []
top1_correct_total = 0
top3_correct_total = 0
total_evaluated_total = 0

metadata_files = glob.glob(os.path.join(metadata_folder, "*.xlsx"))
gt_files = glob.glob(os.path.join(ground_truth_folder, "*.csv"))


for metadata_file in metadata_files:
    match = re.search(r"(SP\d+_\d+)", metadata_file)
    if not match:
        print(f"Could not parse identifier from {metadata_file}, skipping.")
        continue
    identifier = match.group(1)

    gt_file = next(
        (f for f in gt_files if identifier in f and "-uri" in f.lower()),
        None
    )
    if not gt_file:
        print(f"No ground-truth file found for {metadata_file}, skipping.")
        continue

    print(f"Processing {metadata_file} with ground-truth {gt_file}")

    metadata = pd.read_excel(metadata_file, header=[1])


    metadata = metadata.rename(columns={
        '1. Reference Code(s)': 'Reference_Code(s)',
        '3.1. Date of Creation': 'Date_of_Creation',
        'Sender [No Spaces]': 'Sender',
        'Recipient [No Spaces]': 'Recipient',
        'Issued at \n[No Spaces]': 'Issued_at',
        'Sent to \n[No Spaces]': 'Sent_to'
    })


    metadata_df = metadata[['Reference_Code(s)', 'Date_of_Creation', 'Sender', 'Recipient', 'Issued_at', 'Sent_to']]

    metadata_df['Date_of_Creation'] = pd.to_datetime(metadata_df['Date_of_Creation'], errors='coerce')


    gt_file_path = os.path.join(ground_truth_folder, f"{identifier}-uri.csv")
    if not os.path.exists(gt_file_path):
        print(f"Ground truth file not found: {gt_file_path}. Skipping this document.")
        continue
    gt_df = pd.read_csv(gt_file_path)
    gt_df_indexed = gt_df.set_index('Sheet Name')


    results = []
    for idx, doc_row in metadata_df.iterrows():
        doc_result = {'document_id': doc_row['Reference_Code(s)'], 'person_matches': {}}
        for field in ['Sender','Recipient']:
            person_name = doc_row.get(field)
            if pd.isna(person_name) or person_name == '':
                doc_result['person_matches'][field] = []
                continue

            candidates = generate_candidates(person_name, doc_row['Date_of_Creation'], doc_row.get('Issued_at'), doc_row.get('Sent_to'))

            doc_result['person_matches'][field] = candidates[:5]
        results.append(doc_result)


    top1_correct = 0
    top3_correct = 0
    total_evaluated = 0

    metadata_df_indexed = metadata_df.set_index('Reference_Code(s)')

    for r in results:
        doc_code = r['document_id']
        try:
            doc_metadata_row = metadata_df_indexed.loc[doc_code]
        except KeyError:
            continue

        for field in ['Sender','Recipient']:
            person_name = doc_metadata_row.get(field)
            if pd.isna(person_name) or person_name == '':
                continue

            predicted_list = r['person_matches'].get(field, [])
            if not predicted_list:
                continue

            top_candidate_uri = predicted_list[0]['person_uri']

            try:
                gt_person_row = gt_df_indexed.loc[person_name]
                gt_uri = gt_person_row['URI']
            except KeyError:
                continue

            total_evaluated += 1
            if top_candidate_uri == gt_uri:
                top1_correct += 1



            top3_candidate_uris = [
              candidate['person_uri']
              for candidate in predicted_list[:3]
            ]

            if gt_uri in top3_candidate_uris:
              top3_correct += 1


    print(f"Document {identifier}: Top-1 Accuracy = {top1_correct / total_evaluated if total_evaluated > 0 else 0:.2%}")
    print(f"Document {identifier}: Top-3 Accuracy = {top3_correct / total_evaluated if total_evaluated > 0 else 0:.2%}")


    top1_correct_total += top1_correct
    top3_correct_total += top3_correct
    total_evaluated_total += total_evaluated


    all_results.extend(results)


overall_top1_accuracy = top1_correct_total / total_evaluated_total if total_evaluated_total > 0 else 0
overall_top3_accuracy = top3_correct_total / total_evaluated_total if total_evaluated_total > 0 else 0
print(f"\nOverall Top-1 Accuracy across all documents: {overall_top1_accuracy:.2%}, Overall Top-3 Accuracy across all documents: {overall_top3_accuracy:.2%}")


with open(output_json, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"All candidate matches saved to {output_json}")

####Neural entity linking method

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BERT_MODEL = 'bert-base-uncased'
MAX_LEN = 64

metadata_folder = "Metadata/"
ground_truth_folder = "Ground-Truth/"
output_json = "bert_candidate_ranking_all_docs_test.json"

print("Using device:", DEVICE)


kg_df = pd.read_csv("people_entities.csv")

kg_df['floruit_lower'] = pd.to_datetime(kg_df['floruit_lower'], errors='coerce')
kg_df['floruit_upper'] = pd.to_datetime(kg_df['floruit_upper'], errors='coerce')

kg_df['birth_year'] = kg_df['floruit_lower'].dt.year.fillna(0).astype(int)
kg_df['death_year'] = kg_df['floruit_upper'].dt.year.fillna(0).astype(int)


tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL).to(DEVICE)
bert_model.eval()

class EntityLinkingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.similarity = nn.CosineSimilarity(dim=1)

    def forward(self, doc_emb, kg_emb):
        return self.similarity(doc_emb, kg_emb)

model = EntityLinkingModel().to(DEVICE)
model.eval()

def encode_text_batch(text_list, batch_size=64):
    all_embeddings = []

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=MAX_LEN
        )

        input_ids = encoded['input_ids'].to(DEVICE)
        attention_mask = encoded['attention_mask'].to(DEVICE)

        with torch.no_grad():
            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
            cls_emb = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_emb)

    return torch.cat(all_embeddings, dim=0)


print("Precomputing KG embeddings...")

kg_texts = [
    f"{row['name']} {row.get('birth_places','')} {row.get('death_places','')} "
    f"birth year: {row['birth_year']} death year: {row['death_year']}"
    for _, row in kg_df.iterrows()
]

kg_embeddings = encode_text_batch(kg_texts)

all_results = []
top1_correct_total = 0
top3_correct_total = 0
total_evaluated_total = 0

metadata_files = glob.glob(os.path.join(metadata_folder, "*.xlsx"))
gt_files = glob.glob(os.path.join(ground_truth_folder, "*.csv"))

for metadata_file in metadata_files:

    match = re.search(r"(SP\d+_\d+)", metadata_file)
    if not match:
        continue
    identifier = match.group(1)

    gt_file = next(
        (f for f in gt_files if identifier in f and "-uri" in f.lower()),
        None
    )
    if not gt_file:
        continue

    print(f"Processing {identifier}")

    metadata = pd.read_excel(metadata_file, header=[1])

    metadata = metadata.rename(columns={
        '1. Reference Code(s)': 'Reference_Code(s)',
        '3.1. Date of Creation': 'Date_of_Creation',
        'Sender [No Spaces]': 'Sender',
        'Recipient [No Spaces]': 'Recipient',
        'Issued at \n[No Spaces]': 'Issued_at',
        'Sent to \n[No Spaces]': 'Sent_to'
    })

    metadata_df = metadata[['Reference_Code(s)', 'Date_of_Creation',
                             'Sender', 'Recipient', 'Issued_at', 'Sent_to']]

    metadata_df['Date_of_Creation'] = pd.to_datetime(
        metadata_df['Date_of_Creation'], errors='coerce'
    )

    gt_df = pd.read_csv(gt_file)
    gt_df_indexed = gt_df.set_index('Sheet Name')

    metadata_df_indexed = metadata_df.set_index('Reference_Code(s)')

    results = []

    for _, doc in metadata_df.iterrows():

        doc_result = {
            'document_id': doc['Reference_Code(s)'],
            'person_matches': {}
        }

        for field in ['Sender', 'Recipient']:

            person_name = doc.get(field)
            if pd.isna(person_name) or person_name == '':
                doc_result['person_matches'][field] = []
                continue

            doc_year = doc['Date_of_Creation']
            doc_year_str = str(doc_year.year) if pd.notna(doc_year) else ""

            doc_text = f"{person_name} {doc.get('Issued_at','')} " \
                       f"{doc.get('Sent_to','')} document create year: {doc_year_str}"

            doc_emb = encode_text_batch([doc_text])
            doc_emb_expanded = doc_emb.repeat(len(kg_df), 1)

            with torch.no_grad():
                scores = model(doc_emb_expanded, kg_embeddings)

            candidate_scores = [
                {
                    'person_uri': kg_df.iloc[i]['person_uri'],
                    'name': kg_df.iloc[i]['name'],
                    'score': scores[i].item()
                }
                for i in range(len(kg_df))
            ]

            top_candidates = sorted(
                candidate_scores,
                key=lambda x: x['score'],
                reverse=True
            )[:5]

            doc_result['person_matches'][field] = top_candidates

        results.append(doc_result)


    top1_correct = 0
    top3_correct = 0
    total_evaluated = 0

    for r in results:
        doc_id = r['document_id']

        try:
            doc_row = metadata_df_indexed.loc[doc_id]
        except KeyError:
            continue

        for field in ['Sender', 'Recipient']:

            person_name = doc_row.get(field)
            if pd.isna(person_name) or person_name == '':
                continue

            predicted_list = r['person_matches'].get(field, [])
            if not predicted_list:
                continue

            top_uri = predicted_list[0]['person_uri']

            try:
                gt_uri = gt_df_indexed.loc[person_name]['URI']
            except KeyError:
                continue

            total_evaluated += 1
            if top_uri == gt_uri:
                top1_correct += 1

            top3_candidate_uris = [
              candidate['person_uri']
              for candidate in predicted_list[:3]
            ]

            if gt_uri in top3_candidate_uris:
              top3_correct += 1


    print(f"Document {identifier}: Top-1 Accuracy = {top1_correct / total_evaluated if total_evaluated > 0 else 0:.2%}")
    print(f"Document {identifier}: Top-3 Accuracy = {top3_correct / total_evaluated if total_evaluated > 0 else 0:.2%}")

    top1_correct_total += top1_correct
    top3_correct_total += top3_correct
    total_evaluated_total += total_evaluated
    all_results.extend(results)

overall_top1_accuracy = top1_correct_total / total_evaluated_total if total_evaluated_total > 0 else 0
overall_top3_accuracy = top3_correct_total / total_evaluated_total if total_evaluated_total > 0 else 0
print(f"\nOverall Top-1 Accuracy across all documents: {overall_top1_accuracy:.2%}, Overall Top-3 Accuracy across all documents: {overall_top3_accuracy:.2%}")

with open(output_json, "w") as f:
    json.dump(all_results, f, indent=2)

print("All results saved to", output_json)